# Plant Disease Classification using Transfer Learning with EfficientNetB0

## Objective
Build a deep learning model that classifies plant leaf images into their exact disease/healthy class.

The final prediction should return both:
1. **Status**: Healthy, **Diseased**, or **Uncertain** (see prediction helper, confidence threshold)
2. **Class**: the exact class name, for example `Tomato___Late_blight`

## Dataset
The dataset is organized using one folder per class. This format is suitable for `tf.keras.utils.image_dataset_from_directory()`.

## Model
This notebook uses transfer learning with **EfficientNetB0**, data augmentation, validation splitting, evaluation (including confusion matrices and optional binary Healthy/Diseased metrics), and export of metrics and figures suitable for a **project report**.

## Running order (submission / grading)
Run all cells **sequentially** after configuring paths in **section 1** and mounting Google Drive when using Colab. Key stages: **2** paths and extract, **3** data quality, **4** near-duplicate discussion (optional), **5–11** data loading through test evaluation, **12–14** detailed test analysis, **15–16** visualization and deployment helpers, **17** save artifacts, **18** reload check, **19–20** real-world guidance and optional phone-photo batch, **21** printable summary. **Section 3** (exact duplicate scan) is strongly recommended before training.


## 1. Environment setup and configuration

All imports, random seeds, `pathlib.Path` locations, image/batch sizes, and epoch budgets live here.

- **Google Colab**: defaults assume the dataset ZIP lives under `PROJECT_ROOT` on Drive and extracts to `EXTRACT_PATH`.
- **Local runs**: set `PROJECT_ROOT` and `EXTRACT_PATH` to folders on your machine (skip Drive mount when not in Colab).


In [ ]:
import json
import warnings
import shutil
import zipfile
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import load_img, img_to_array

from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight

from PIL import Image, UnidentifiedImageError


def get_health_status(class_name: str) -> str:
    """PlantVillage-style label: '<Crop>___<condition>'. Healthy iff condition == 'healthy'."""
    if "___" not in class_name:
        return "Diseased"
    _crop, disease_part = class_name.split("___", 1)
    return "Healthy" if disease_part == "healthy" else "Diseased"


# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# ---------------------------------------------------------------------------
# Data loading & training hyperparameters
# ---------------------------------------------------------------------------
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
# If True, pass sklearn balanced class_weight dict to both model.fit stages (see class-weight cell after label counts).
USE_CLASS_WEIGHTS = False
HEAD_EPOCHS = 15
FINE_TUNE_EPOCHS = 10

# ---------------------------------------------------------------------------
# Paths (Colab + Google Drive defaults)
# Adjust PROJECT_ROOT / EXTRACT_PATH when running outside Colab.
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/plant_project")
ZIP_PATH = PROJECT_ROOT / "archive.zip"
EXTRACT_PATH = Path("/content/dataset")
TRAIN_DIR = EXTRACT_PATH / "dataset" / "train"
TEST_DIR = EXTRACT_PATH / "dataset" / "test"
MODEL_SAVE_DIR = PROJECT_ROOT
CHECKPOINT_PATH = MODEL_SAVE_DIR / "best_plant_disease_model.keras"
# Optional folder of real-world phone photos (not used in training). Create on Drive to run section 20.
PHONE_TEST_DIR = PROJECT_ROOT / "phone_test_photos"

# Exact train/test duplicate handling (SHA-256): see section 3. Only files under TEST_DIR may be removed.
REMOVE_DUPLICATE_TEST_FILES = True

# Final export filenames (under MODEL_SAVE_DIR)
KERAS_MODEL_FILENAME = "plant_disease_efficientnet_model.keras"
MODEL_DISPLAY_NAME = Path(KERAS_MODEL_FILENAME).stem
CLASS_NAMES_FILENAME = "class_names.json"
METADATA_FILENAME = "model_metadata.json"
METRICS_FILENAME = "metrics.json"
TRAINING_HISTORY_FILENAME = "training_history.json"
CLASSIFICATION_REPORT_CSV = "classification_report.csv"
CONFUSION_MATRIX_CSV = "confusion_matrix.csv"
CONFUSION_MATRIX_NORM_CSV = "normalized_confusion_matrix.csv"

# tf.data pipeline (see section 7)
# On-disk cache under Colab's local /content avoids RAM-only .cache() OOM on large datasets.
# Set to None to disable file caching (more CPU/IO each epoch, minimal disk use).
TF_DATA_CACHE_DIR = Path("/content/tf_data_cache")
# Shuffle buffer is capped by training set size in section 7; this is the upper cap.
SHUFFLE_BUFFER_MAX = 32768

print(f"TensorFlow version: {tf.__version__}")
_gpus = tf.config.list_physical_devices("GPU")
print(f"GPU devices available: {len(_gpus)} —", _gpus if _gpus else "(none, training will use CPU)")


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Google Colab — skipping drive.mount(). Ensure PROJECT_ROOT and dataset paths exist.")


In [ ]:
if not ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset ZIP not found at {ZIP_PATH}. "
        "Upload archive.zip to PROJECT_ROOT on Google Drive (or update ZIP_PATH / PROJECT_ROOT)."
    )

# Fresh extract: remove only EXTRACT_PATH so stale files from a prior run cannot remain.
if EXTRACT_PATH.exists():
    print(f"Removing previous extraction directory: {EXTRACT_PATH}")
    shutil.rmtree(EXTRACT_PATH)

EXTRACT_PATH.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Done extracting.")
top_level = sorted(EXTRACT_PATH.iterdir(), key=lambda p: p.name.lower())
print("Top-level folders/files under EXTRACT_PATH:")
for p in top_level:
    label = "dir" if p.is_dir() else "file"
    print(f"  [{label}] {p.name}")

if not TRAIN_DIR.is_dir():
    raise FileNotFoundError(
        f"Expected training directory missing after extract: {TRAIN_DIR}. "
        f"The ZIP should unpack so that `{TRAIN_DIR.name}` exists under {EXTRACT_PATH / 'dataset'} (see section 2 folder layout)."
    )
if not TEST_DIR.is_dir():
    raise FileNotFoundError(
        f"Expected test directory missing after extract: {TEST_DIR}. "
        f"The ZIP should unpack so that `{TEST_DIR.name}` exists under {EXTRACT_PATH / 'dataset'} (see section 2 folder layout)."
    )
print("OK: TRAIN_DIR and TEST_DIR are present.")


## 2. Dataset paths

The extraction cell above removes the previous **`EXTRACT_PATH`** tree (and only that path) before unpacking, so stale files from an older run cannot remain. It also checks that **`TRAIN_DIR`** and **`TEST_DIR`** exist after extraction.

Make sure the ZIP file is fully extracted before training. The expected structure is:

```text
dataset/
  dataset/
    train/
      class_1/
      class_2/
      ...
    test/
      class_1/
      class_2/
      ...
```


In [ ]:
print(f"Training directory exists: {TRAIN_DIR.is_dir()} — {TRAIN_DIR.resolve()}")
print(f"Test directory exists:     {TEST_DIR.is_dir()} — {TEST_DIR.resolve()}")

if not TRAIN_DIR.is_dir():
    raise FileNotFoundError(f"Training folder not found: {TRAIN_DIR}")

if not TEST_DIR.is_dir():
    raise FileNotFoundError(f"Test folder not found: {TEST_DIR}")


## 3. Data quality: corrupted files and train/test duplicates

We scan image files with **PIL** (corrupted / unreadable) and **SHA-256** content hashes. Corrupted images are **not** deleted here — review printed paths and fix bad files manually if needed.

**Exact duplicates (train vs test):** We compare **SHA-256** hashes of every train and test image. **Duplicate test files are removed when `REMOVE_DUPLICATE_TEST_FILES` is True** (default in section 1), because byte-identical copies of training images sit in **both** splits and **inflate test accuracy** (the model can match pixels it already memorized). Only paths under **`TEST_DIR`** are deleted — **training files are never removed**. If **`REMOVE_DUPLICATE_TEST_FILES`** is **False** and duplicates exist, the next cell **raises `RuntimeError`** and stops so you cannot train or evaluate on a leaky split.

**Limit of SHA-256:** Matching hashes only mean the files are **identical on disk** (same bytes). They do **not** detect *near-duplicates* — for example the same photo resaved with different JPEG quality, resized, cropped, or slightly recolored will usually hash differently.

Identical files in **train** and **test** cause **data leakage**: the model may see the same image content during training and again at test time, which **inflates test accuracy**.


In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def list_image_paths(dataset_root: Path):
    '''All image paths under class subfolders (same extensions as the training pipeline).'''
    root = Path(dataset_root)
    paths = []
    for class_path in sorted(p for p in root.iterdir() if p.is_dir()):
        for fp in class_path.iterdir():
            if fp.is_file() and fp.suffix.lower() in IMAGE_EXTS:
                paths.append(fp)
    return paths


def find_corrupted_images_pil(image_paths):
    '''Return (path, error) tuples for images PIL cannot verify/load. Does not delete files.'''
    bad = []
    for p in image_paths:
        try:
            with Image.open(p) as img:
                img.verify()
            with Image.open(p) as img:
                img.convert("RGB")
                img.load()
        except (OSError, UnidentifiedImageError, ValueError) as e:
            bad.append((p, repr(e)))
        except Image.DecompressionBombError as e:
            bad.append((p, repr(e)))
    return bad


def sha256_file(path, chunk_bytes=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk_bytes), b""):
            h.update(block)
    return h.hexdigest()


def invert_hash_map(path_list):
    '''hash -> list of paths (identical file bytes may appear more than once).'''
    hmap = {}
    for p in path_list:
        try:
            digest = sha256_file(p)
        except OSError as e:
            print(f"Skipping hash (read error): {p} — {e}")
            continue
        hmap.setdefault(digest, []).append(p)
    return hmap


def _is_strictly_under(path: Path, parent: Path) -> bool:
    """True if path is inside parent (not parent itself)."""
    path = path.resolve()
    parent = parent.resolve()
    try:
        path.relative_to(parent)
    except ValueError:
        return False
    return path != parent


train_image_paths = list_image_paths(TRAIN_DIR)
test_image_paths = list_image_paths(TEST_DIR)

corrupt_train = find_corrupted_images_pil(train_image_paths)
corrupt_test = find_corrupted_images_pil(test_image_paths)

print("=== Corrupted images (PIL) ===")
print(f"Train: {len(corrupt_train)} corrupted")
print(f"Test:  {len(corrupt_test)} corrupted")
if corrupt_train:
    print("Example train:", corrupt_train[0])
if corrupt_test:
    print("Example test:", corrupt_test[0])

train_by_hash = invert_hash_map(train_image_paths)

while True:
    test_image_paths = list_image_paths(TEST_DIR)
    test_by_hash = invert_hash_map(test_image_paths)
    train_hashes = set(train_by_hash.keys())
    test_hashes = set(test_by_hash.keys())
    shared_hashes = train_hashes & test_hashes

    duplicate_test_files = []
    for h in shared_hashes:
        duplicate_test_files.extend(test_by_hash[h])

    print("\n=== Duplicate content (train vs test, SHA-256) ===")
    print(f"SHA-256 hashes shared between train and test: {len(shared_hashes)}")
    n_dup_test = len(duplicate_test_files)
    print(f"Number of duplicate test files (byte-identical to at least one train file): {n_dup_test}")

    if not duplicate_test_files:
        print(
            "\nOK: No train/test SHA-256 duplicates — no test file is an exact byte copy of a training file "
            "(among successfully hashed images). Safe to continue."
        )
        break

    print("\nExample duplicate test paths:")
    for p in duplicate_test_files[:5]:
        print(f"  {p}")
    if n_dup_test > 5:
        print(f"  ... and {n_dup_test - 5} more")

    if REMOVE_DUPLICATE_TEST_FILES:
        removed = 0
        for p in duplicate_test_files:
            p = Path(p)
            if not _is_strictly_under(p, TEST_DIR):
                raise RuntimeError(
                    f"Refusing to delete path outside TEST_DIR: {p} (TEST_DIR={TEST_DIR.resolve()})"
                )
            p.unlink()
            removed += 1
        print(
            f"\nRemoved {removed} duplicate test file(s) under TEST_DIR (same SHA-256 as a training image). "
            "Re-running duplicate check..."
        )
        continue

    raise RuntimeError(
        f"Stopping: {n_dup_test} test image(s) are byte-identical to training images (SHA-256). "
        "Set REMOVE_DUPLICATE_TEST_FILES = True in section 1 to drop only those test copies, "
        "or fix the dataset split manually."
    )


## 4. Near-duplicate risk (optional)

The **SHA-256** check above only finds **exact duplicate files** (identical bytes on disk). It does **not** catch **near-duplicates**: the **same or almost the same photograph** stored as a different file after **resizing**, **JPEG recompression**, **cropping**, **color tweaks**, **offline augmentation** exports, or multiple shots of the **same leaf**. Those can still appear in **train**, **validation**, and **test** at once, so **metrics can look better than true generalization** even when no cryptographic duplicates exist.

Treat exact-hash deduping as a **baseline hygiene** step, not proof of a clean split. Stronger workflows use **perceptual** similarity or **embedding** neighbors (see the TODO comments in the next cell for directions—**not wired up** in this notebook yet).


In [ ]:
# -----------------------------------------------------------------------------
# TODO (future improvement): near-duplicate detection beyond exact SHA-256
# Documentation only — no new libraries installed here; this cell does not alter training.
# -----------------------------------------------------------------------------

# TODO 1) Perceptual hashing (e.g. `imagehash` in PyPI: phash / average_hash / dhash)
#   - Decode each image to a small standard size, compute a compact fingerprint.
#   - Compare Hamming distance between hashes; pairs (or clusters) below a threshold are
#     candidates for near-duplicates even when file bytes differ (resize, re-encode, etc.).
#   - Optionally index by hash prefix or use LSH-style bucketing for large corpora.

# TODO 2) Image embeddings
#   - Encode each image with a frozen pretrained CNN / ViT (or similar) to a fixed-length vector.
#   - Use cosine similarity or k-NN (e.g. scikit-learn, FAISS) to flag high-similarity pairs
#     across train vs validation vs test; manually review borderline matches to set thresholds.

# TODO 3) Group augmented variants before splitting
#   - When the pipeline creates rotations/crops/exports from one original, assign a shared
#     **group id** (e.g. from original filename stem or UUID) and **keep all variants in one
#     split only** (train XOR val XOR test) via grouped splits or hashing the group id into a fold.
#   - Prevents trivially related views of the same scene from leaking across splits.

pass  # no-op: valid Python; nothing runs until you implement the ideas above.


## 5. Check class distribution

This helps you see whether the dataset is balanced. If some classes have many fewer images, the model may perform worse on those classes.


In [ ]:
def count_images_per_class(directory: Path):
    rows = []
    root = Path(directory)
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    for class_path in sorted(p for p in root.iterdir() if p.is_dir()):
        image_files = [f for f in class_path.iterdir() if f.is_file() and f.suffix.lower() in exts]
        rows.append({"class": class_path.name, "count": len(image_files)})
    return pd.DataFrame(rows).sort_values("count", ascending=True)

train_counts = count_images_per_class(TRAIN_DIR)
test_counts = count_images_per_class(TEST_DIR)

print("Training set class distribution:")
try:
    display(train_counts)  # Colab / Jupyter
except NameError:
    print(train_counts.to_string(index=False))

print("\nTest set class distribution:")
try:
    display(test_counts)
except NameError:
    print(test_counts.to_string(index=False))


In [ ]:
plt.figure(figsize=(12, max(6, len(train_counts) * 0.3)))
plt.barh(train_counts["class"], train_counts["count"])
plt.title("Training Images per Class")
plt.xlabel("Number of images")
plt.ylabel("Class")
plt.tight_layout()
plt.show()


## 6. Load train, validation, and test datasets

The training folder is split into training and validation subsets. The test folder is kept separate and is not shuffled so evaluation stays aligned with labels.

Section **5** counts images **on disk** under `train/` and `test/`. After `train_data`, `val_data`, and `test_data` are created below, the **next code cell** counts **labels inside the real `tf.data` pipelines** (the actual 80/20 split and the test loader). Missing classes in **training** or **test** stop the notebook with an error; missing classes **only in validation** trigger a **warning** (can happen when a class has very few images and the fixed split leaves none in the validation subset).


In [ ]:
# Training images are split into train (80%) and validation (20%) with the same seed.
# The separate `test/` folder is never used during fit — no leakage from test into train/val.
train_data = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
)

val_data = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
)

test_data = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,  # stable order for metrics and confusion matrix
)

class_names = train_data.class_names
num_classes = len(class_names)

# Labels must map to the same string classes on train and test (alphabetical folder order).
if test_data.class_names != class_names:
    raise ValueError(
        "Train and test class_names differ — evaluation labels will be wrong. "
        f"Train ({len(class_names)}): {class_names[:5]}... "
        f"Test ({len(test_data.class_names)}): {test_data.class_names[:5]}..."
    )

print("Number of classes:", num_classes)
print("Classes:")
for i, name in enumerate(class_names):
    print(i, name)


In [ ]:
def count_labels_in_tf_dataset(ds, class_names, split_name="split"):
    """Count integer labels by iterating every batch in the dataset (one full pass)."""
    counts = np.zeros(len(class_names), dtype=np.int64)
    n_batches = 0
    for _, labels in ds:
        for lab in labels.numpy().ravel():
            counts[int(lab)] += 1
        n_batches += 1
    print(f"  {split_name}: scanned {n_batches} batch(es), total samples {int(counts.sum())}")
    return {class_names[i]: int(counts[i]) for i in range(len(class_names))}


def label_counts_to_dataframe(count_map):
    """One row per class, sorted by class name."""
    return (
        pd.DataFrame(
            {"class": list(count_map.keys()), "count": list(count_map.values())}
        )
        .sort_values("class", ignore_index=True)
    )


def classes_missing_from_split(count_map):
    """Class names with zero samples in this split."""
    return sorted([name for name, c in count_map.items() if c == 0])


print("Counting labels in TensorFlow datasets (one full pass per split)...\n")
train_label_counts = count_labels_in_tf_dataset(train_data, class_names, "train (tf.data)")
val_label_counts = count_labels_in_tf_dataset(val_data, class_names, "validation (tf.data)")
test_label_counts = count_labels_in_tf_dataset(test_data, class_names, "test (tf.data)")

df_train_labels = label_counts_to_dataframe(train_label_counts)
df_val_labels = label_counts_to_dataframe(val_label_counts)
df_test_labels = label_counts_to_dataframe(test_label_counts)

print("\nActual training split — label counts:")
try:
    display(df_train_labels)
except NameError:
    print(df_train_labels.to_string(index=False))

print("\nActual validation split — label counts:")
try:
    display(df_val_labels)
except NameError:
    print(df_val_labels.to_string(index=False))

print("\nActual test split — label counts:")
try:
    display(df_test_labels)
except NameError:
    print(df_test_labels.to_string(index=False))

missing_train = classes_missing_from_split(train_label_counts)
missing_val = classes_missing_from_split(val_label_counts)
missing_test = classes_missing_from_split(test_label_counts)

if missing_train:
    raise RuntimeError(
        "Training tf.data split has no samples for the following class(es): "
        f"{missing_train}. You cannot train a multi-class head without at least one "
        "example per class in the training subset — add images or change validation_split/SEED."
    )

if missing_test:
    raise RuntimeError(
        "Test tf.data split has no samples for the following class(es): "
        f"{missing_test}. The held-out test set must include every class for a complete "
        "benchmark — fix the test folder or labels before continuing."
    )

if missing_val:
    warnings.warn(
        "Validation tf.data split has no samples for the following class(es): "
        f"{missing_val}. Validation metrics will not reflect those classes (often due to "
        "very small per-class counts under an 80/20 file split). Consider more images per "
        "class, a stratified split, or adjusting validation_split.",
        UserWarning,
        stacklevel=1,
    )

if not missing_val:
    print("\nOK: Every class appears at least once in train, validation, and test tf.data splits.")
else:
    print("\nOK: Every class appears at least once in train and test tf.data splits (see validation warning above).")


### Class weights (optional)

**When class weights help:** If some diseases (classes) have **many fewer training images** than others, plain cross-entropy pushes the model toward **majority classes**. **`class_weight='balanced'`** (via `sklearn.utils.class_weight.compute_class_weight`) increases the loss contribution from **rare** labels so the head and fine-tune stages both pay more attention to underrepresented classes.

**When you may leave them off:** If classes are already **similar in size**, weights add little; very **aggressive** weighting can **hurt** accuracy on common classes or interact oddly with metrics. Toggle **`USE_CLASS_WEIGHTS`** in section 1: **`False`** means **`class_weight` is not passed** to `model.fit` at all (default behavior).


In [ ]:
# Uses actual training-split counts from the previous cell (not folder totals).
if USE_CLASS_WEIGHTS:
    y_train_idx = np.concatenate(
        [np.full(train_label_counts[name], i, dtype=np.int32) for i, name in enumerate(class_names)]
    )
    cw_arr = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(len(class_names)),
        y=y_train_idx,
    )
    class_weight_for_fit = {i: float(cw_arr[i]) for i in range(len(class_names))}
    df_class_weights = pd.DataFrame(
        {
            "class_index": list(range(len(class_names))),
            "class_name": class_names,
            "train_count": [train_label_counts[c] for c in class_names],
            "weight": [class_weight_for_fit[i] for i in range(len(class_names))],
        }
    )
    print("Balanced class weights for model.fit (train split only):")
    try:
        display(df_class_weights)
    except NameError:
        print(df_class_weights.to_string(index=False))
else:
    class_weight_for_fit = None
    print("USE_CLASS_WEIGHTS is False: model.fit runs without class_weight.")


In [ ]:
plt.figure(figsize=(12, 12))
for images, labels in train_data.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()


## 7. Data augmentation and performance optimization

**Why augment for real-world phone photos?** Dataset images are often centered, well-lit, and uniform. **Field and phone pictures** differ: the leaf is **tilted** (in-plane rotation), **not perfectly framed** (translation / zoom), may appear **mirror-reversed** when snapped from either side (**horizontal flip**), and **lighting** changes **contrast** and **brightness** (sun, shade, flash). **Background** and distance also vary, which **zoom** and **shift** partly mimic. These layers add **mild** randomness so the model learns robust cues instead of memorizing studio-like crops—while staying **conservative** so **small lesions, spots, and discoloration** (disease signals) are not smeared away by extreme warp or color abuse.

Augmentation runs **only during training** (`training=True` inside `model.fit`); validation and test see identity transforms, so metrics stay fair.

**`tf.data`:** Training batches use **on-disk `cache()`** (paths from `TF_DATA_CACHE_DIR` in section 1) so decoded images are not all held in RAM. **Training** uses a **large shuffle buffer** (up to `SHUFFLE_BUFFER_MAX`, capped by dataset size) with **`reshuffle_each_iteration=True`** so each epoch sees a new random order. **Validation** and **test** stay **deterministic** (no shuffle). Delete the cache folder if you change the dataset or preprocessing so stale files are not reused.


In [ ]:
# Random augmentation: only active when training=True (inside model.fit on train_data).
# Ranges kept moderate to reduce over-distortion of fine disease textures (spots, streaks, mild chlorosis).
_aug_layers = [
    layers.RandomFlip("horizontal"),  # phone may capture either leaf face; many diseases are symmetric enough
    layers.RandomRotation(0.07),  # ~±25° — handheld tilt, not full upside-down
    layers.RandomZoom(height_factor=(-0.08, 0.08), width_factor=(-0.08, 0.08)),  # slight distance / crop variation
    layers.RandomTranslation(height_factor=(-0.06, 0.06), width_factor=(-0.06, 0.06)),  # leaf not always centered
    layers.RandomContrast(0.08),  # outdoor vs indoor / shadow edges (mild)
]

_RandomBrightness = getattr(layers, "RandomBrightness", None)
if _RandomBrightness is not None:
    try:
        # TF 2.10+ / Keras preprocessing: value_range matches float RGB in [0, 255] from image_dataset_from_directory
        _aug_layers.append(_RandomBrightness(0.1, value_range=(0.0, 255.0)))
    except TypeError:
        try:
            _aug_layers.append(_RandomBrightness(factor=0.1, value_range=(0.0, 255.0)))
        except Exception as _e:
            print(f"RandomBrightness omitted (constructor mismatch: {_e}).")
else:
    print("RandomBrightness not in tf.keras.layers for this TensorFlow build; contrast aug still covers some lighting shift.")

data_augmentation = tf.keras.Sequential(_aug_layers, name="data_augmentation")

AUTOTUNE = tf.data.AUTOTUNE

# File-backed cache: RAM-only .cache() can exhaust memory on large image pipelines.
# First epoch writes these files; later epochs reuse them for faster input throughput.
if TF_DATA_CACHE_DIR is not None:
    TF_DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    _train_cache = str(TF_DATA_CACHE_DIR / "train_dataset.cache")
    _val_cache = str(TF_DATA_CACHE_DIR / "val_dataset.cache")
    _test_cache = str(TF_DATA_CACHE_DIR / "test_dataset.cache")
    train_data = train_data.cache(_train_cache)
    val_data = val_data.cache(_val_cache)
    test_data = test_data.cache(_test_cache)

# Shuffle buffer is in batches (dataset is already batched). Cap by cardinality when known.
_card_train = int(tf.data.experimental.cardinality(train_data).numpy())
if _card_train >= 0:
    shuffle_buffer = min(SHUFFLE_BUFFER_MAX, max(_card_train, 1))
else:
    shuffle_buffer = min(SHUFFLE_BUFFER_MAX, 10000)

# Training: new permutation each epoch; fixed seed keeps runs comparable.
train_data = train_data.shuffle(
    shuffle_buffer,
    seed=SEED,
    reshuffle_each_iteration=True,
).prefetch(buffer_size=AUTOTUNE)

# Validation: no shuffle — order is stable across epochs for consistent val metrics.
val_data = val_data.prefetch(buffer_size=AUTOTUNE)

# Test: loader already has shuffle=False; no shuffle here — reproducible evaluate/predict.
test_data = test_data.prefetch(buffer_size=AUTOTUNE)


## 8. Build the EfficientNetB0 transfer learning model

EfficientNetB0 is pretrained on ImageNet. Stage 1 freezes the base and trains the head. Stack: **Input → augmentation → EfficientNetB0 → GlobalAveragePooling2D → BatchNorm → Dropout → Dense → Dropout → softmax**.


In [ ]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE, 3),
)

base_model.trainable = False

# tf.keras.applications.EfficientNetB0 applies the correct ImageNet preprocessing inside the model.
# Do not insert a manual Rescaling layer to [-1, 1] — that would be wrong for this API.
# image_dataset_from_directory yields float32 RGB in [0, 255], which is what this base expects.
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(*IMG_SIZE, 3)),
        data_augmentation,
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy"),
    ],
)

model.summary()


In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1,
)

# Save the best model on the validation set (lowest val_loss). Matches EarlyStopping.
checkpoint = ModelCheckpoint(
    str(CHECKPOINT_PATH),
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1,
)


## 9. Training stage 1: train the classification head


In [ ]:
_fit_class_weight = {}
if USE_CLASS_WEIGHTS:
    _fit_class_weight["class_weight"] = class_weight_for_fit

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=HEAD_EPOCHS,
    callbacks=[early_stop, reduce_lr, checkpoint],
    **_fit_class_weight,
)


## 10. Training stage 2: fine-tune the last layers of EfficientNetB0

Fine-tuning usually improves the model because the pretrained feature extractor adapts to plant leaf disease images. **Stage 2** unfreezes only the **last convolutional blocks** of EfficientNetB0 (see code) and recompiles with a **low learning rate** so ImageNet features are adjusted gently.

**Why freeze `BatchNormalization` in the base during fine-tuning?** BN layers track **running mean and variance** (and may train scale/shift). With **small fine-tune batches** and a **new domain** (plant leaves), updating those statistics can be **noisy and unstable**, which sometimes **hurts** transfer more than it helps. Keeping **all** base `BatchNormalization` layers **non-trainable** is a common, **conservative** choice: **convolutional** weights can still adapt to disease textures, while BN keeps **stable** behavior tied to the pretrained normalization, **reducing** the risk of **overfitting** the normalization stats to the mini-batch distribution during fine-tune.


In [ ]:
base_model.trainable = True

# Freeze early layers; train only the last N EfficientNet blocks (conv weights can adapt to leaves).
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep every BatchNorm in the base frozen: safer stats when fine-tuning with small batches (see section 10 markdown).
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy"),
    ],
)

_fit_class_weight_ft = {}
if USE_CLASS_WEIGHTS:
    _fit_class_weight_ft["class_weight"] = class_weight_for_fit

fine_tune_history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=[early_stop, reduce_lr, checkpoint],
    **_fit_class_weight_ft,
)


## 11. Evaluate on the separate test set

We **reload `ModelCheckpoint`** (best **validation loss**) into `best_model`, then compute test **loss**, **top-1 accuracy**, and **top-3 accuracy** on that checkpoint — not on the last training epoch by default.


In [ ]:
# Load the best validation weights from disk so test numbers match the saved checkpoint
# (not merely the last epoch in memory).
print("Loading best validation model from checkpoint...")
best_model = tf.keras.models.load_model(str(CHECKPOINT_PATH))

test_metrics = best_model.evaluate(test_data, verbose=1, return_dict=True)
print(f"Test loss: {test_metrics.get('loss', float('nan')):.4f}")
if "accuracy" in test_metrics:
    print(f"Test accuracy: {test_metrics['accuracy']:.4f}")
if "top3_accuracy" in test_metrics:
    print(f"Test top-3 accuracy: {test_metrics['top3_accuracy']:.4f}")


In [ ]:
def combine_histories(*histories):
    combined = {}
    for h in histories:
        if h is None:
            continue
        for key, values in h.history.items():
            combined.setdefault(key, []).extend(values)
    return combined

combined_history = combine_histories(history, fine_tune_history)


def _plot_metric_pair(ax, history_dict, train_key, val_key, title):
    """Plot train/val series if present; tolerate missing keys or length mismatch."""
    y_train = history_dict.get(train_key) or []
    y_val = history_dict.get(val_key) or []
    plotted = False
    if y_train:
        ax.plot(range(1, len(y_train) + 1), y_train, label=train_key)
        plotted = True
    if y_val:
        ax.plot(range(1, len(y_val) + 1), y_val, label=val_key)
        plotted = True
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    if plotted:
        ax.legend()
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)


fig, axes = plt.subplots(1, 3, figsize=(16, 5))
_plot_metric_pair(axes[0], combined_history, "accuracy", "val_accuracy", "Top-1 accuracy")
_plot_metric_pair(
    axes[1], combined_history, "top3_accuracy", "val_top3_accuracy", "Top-3 accuracy"
)
_plot_metric_pair(axes[2], combined_history, "loss", "val_loss", "Loss")
plt.tight_layout()
plt.show()

# Diagnose fit: large train–val gap in accuracy + rising val loss often indicates overfitting.
# If train and val track closely but both stay low, consider more data, stronger aug, or longer head training.


## 12. Confusion matrix and classification report

**Raw counts:** Each cell is the **number of test images** whose **true** class is the row and **predicted** class is the column. Large datasets make dominant classes visually heavy even when the **rate** of error is low.

**Row-normalized (`normalize='true'`):** Each **row** (true class) sums to **1**. Diagonal cells are **recall** for that class; off-diagonals show **what fraction** of that true class was confused with each predicted label—useful for comparing **error patterns** across rare and frequent classes.


In [ ]:
# Final test evaluation: use this held-out set only once for the report below.
# Do not tune hyperparameters using test results and re-train — that leaks test information.

y_true = np.concatenate([labels.numpy() for _, labels in test_data], axis=0).astype(int)
y_pred_probs = best_model.predict(test_data, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)
bal_acc = balanced_accuracy_score(y_true, y_pred)
print("Test metrics (sklearn):")
print(f"  Macro F1:        {f1_macro:.4f}")
print(f"  Weighted F1:     {f1_weighted:.4f}")
print(f"  Balanced acc:    {bal_acc:.4f}")

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(30, 28))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=90, cmap="Blues", values_format="d", colorbar=False)
plt.setp(ax.get_xticklabels(), rotation=90, ha="center", fontsize=8)
plt.setp(ax.get_yticklabels(), fontsize=8)
ax.set_xlabel("Predicted label", fontsize=10)
ax.set_ylabel("True label", fontsize=10)
plt.title("Confusion matrix — raw counts (test set, best val checkpoint)", fontsize=12)
plt.tight_layout()
plt.show()

cm_norm = confusion_matrix(y_true, y_pred, normalize="true")

fig2, ax2 = plt.subplots(figsize=(30, 28))
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names)
disp_norm.plot(
    ax=ax2,
    xticks_rotation=90,
    cmap="Blues",
    values_format=".2f",
    colorbar=True,
)
plt.setp(ax2.get_xticklabels(), rotation=90, ha="center", fontsize=8)
plt.setp(ax2.get_yticklabels(), fontsize=8)
ax2.set_xlabel("Predicted label", fontsize=10)
ax2.set_ylabel("True label", fontsize=10)
plt.title(
    "Confusion matrix — row-normalized (fraction of each true class; diagonal = recall)",
    fontsize=12,
)
plt.tight_layout()
plt.show()


In [ ]:
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

_acc = report_dict.pop("accuracy", None)
df_classification = pd.DataFrame.from_dict(
    {k: v for k, v in report_dict.items() if isinstance(v, dict)},
    orient="index",
)
df_classification.index.name = "class"
if _acc is not None:
    print(f"Overall accuracy (from report): {_acc:.4f}")
print("\nClassification report (per-class + averages) as DataFrame:")
try:
    display(df_classification)
except NameError:
    print(df_classification.to_string())

_class_rows = df_classification.loc[df_classification.index.isin(class_names)]
df_worst_f1 = _class_rows.nsmallest(10, "f1-score").reset_index()
print("\n10 worst classes by F1-score (ascending):")
try:
    display(df_worst_f1)
except NameError:
    print(df_worst_f1.to_string(index=False))


## 13. Healthy vs Diseased (binary evaluation)

**Exact disease accuracy** (all classes) answers: did we predict the **precise** folder label (e.g. `Tomato___Late_blight`)? **Healthy vs Diseased** accuracy answers a **coarser** question: is the leaf **healthy** or **any disease**? A sample can be **binary-correct** but **multi-class wrong** (e.g. predicted another disease), or **multi-class correct** on disease type but we rarely confuse healthy with disease at top-1. The metrics below use **`get_health_status`**: **Diseased** unless the segment after the **first** `___` is exactly `healthy`.


In [ ]:
bin_labels = ["Healthy", "Diseased"]
y_true_hvd = np.array([get_health_status(class_names[i]) for i in y_true])
y_pred_hvd = np.array([get_health_status(class_names[i]) for i in y_pred])

print("Healthy vs Diseased — classification report (test set):")
print(
    classification_report(
        y_true_hvd,
        y_pred_hvd,
        labels=bin_labels,
        target_names=bin_labels,
        zero_division=0,
    )
)

cm_hvd = confusion_matrix(y_true_hvd, y_pred_hvd, labels=bin_labels)
fig_hvd, ax_hvd = plt.subplots(figsize=(8, 6.5))
disp_hvd = ConfusionMatrixDisplay(confusion_matrix=cm_hvd, display_labels=bin_labels)
disp_hvd.plot(ax=ax_hvd, cmap="Blues", values_format="d", colorbar=False)
plt.setp(ax_hvd.get_xticklabels(), rotation=45, ha="right", fontsize=12)
plt.setp(ax_hvd.get_yticklabels(), fontsize=12)
ax_hvd.set_xlabel("Predicted", fontsize=12)
ax_hvd.set_ylabel("True", fontsize=12)
plt.title("Healthy vs Diseased — confusion matrix (counts, test set)", fontsize=13)
plt.tight_layout()
plt.show()


## 14. Confidence analysis (test set)

The **maximum softmax probability** is a **model score**, not a calibrated probability of being correct. Neural nets are often **overconfident** on wrong predictions (especially under **distribution shift** or **novel** backgrounds). **Do not** treat these values as **medical or agricultural certainty**; use them only as a **relative** ranking or to flag samples that may deserve **human review**.


In [ ]:
_max_conf = np.max(y_pred_probs, axis=1)
_correct = y_pred == y_true

df_confidence = pd.DataFrame(
    {
        "confidence": _max_conf,
        "correct": _correct,
        "true_class": [class_names[i] for i in y_true],
        "predicted_class": [class_names[i] for i in y_pred],
    }
)

_mask_ok = df_confidence["correct"]
_n_wrong = int((~_mask_ok).sum())
print("Confidence summary (max softmax per image, test set):")
if _mask_ok.any():
    print(f"  Mean confidence when correct:   {df_confidence.loc[_mask_ok, 'confidence'].mean():.4f}")
    print(f"  Median confidence when correct: {df_confidence.loc[_mask_ok, 'confidence'].median():.4f}")
else:
    print("  No correct predictions — mean/median for correct N/A.")
if (~_mask_ok).any():
    print(f"  Mean confidence when wrong:     {df_confidence.loc[~_mask_ok, 'confidence'].mean():.4f}")
    print(f"  Median confidence when wrong:   {df_confidence.loc[~_mask_ok, 'confidence'].median():.4f}")
else:
    print("  No wrong predictions — mean/median for wrong N/A.")

print(f"\nTop 20 wrong predictions with highest confidence (of {_n_wrong} errors):")
if _n_wrong > 0:
    _wrong_top20 = df_confidence.loc[~_mask_ok].nlargest(20, "confidence")
    try:
        display(_wrong_top20)
    except NameError:
        print(_wrong_top20.to_string(index=False))
else:
    print("  (none)")

fig_c, ax_c = plt.subplots(figsize=(10, 5))
if _mask_ok.any():
    ax_c.hist(
        df_confidence.loc[_mask_ok, "confidence"],
        bins=40,
        alpha=0.55,
        label="Correct",
        density=True,
        color="tab:green",
    )
if (~_mask_ok).any():
    ax_c.hist(
        df_confidence.loc[~_mask_ok, "confidence"],
        bins=40,
        alpha=0.55,
        label="Wrong",
        density=True,
        color="tab:red",
    )
ax_c.set_xlabel("Max softmax confidence")
ax_c.set_ylabel("Density")
ax_c.set_title("Test set: confidence distribution (correct vs wrong)")
ax_c.legend()
ax_c.set_xlim(0.0, 1.0)
plt.tight_layout()
plt.show()


## 15. Visualize predictions


In [ ]:
plt.figure(figsize=(15, 15))

for images, labels in test_data.take(1):
    preds = best_model.predict(images, verbose=0)
    pred_labels = np.argmax(preds, axis=1)

    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))

        true_label = class_names[int(labels[i])]
        predicted_label = class_names[int(pred_labels[i])]
        confidence = np.max(preds[i]) * 100

        title = f"True: {true_label}\nPred: {predicted_label}\nConf: {confidence:.1f}%"
        plt.title(title, fontsize=9)
        plt.axis("off")

plt.tight_layout()
plt.show()


## 16. Final prediction functions

Use **`best_model`** after training. **`predict_image`** returns **status** (`Healthy`, `Diseased`, or **`Uncertain`** if top-1 confidence is below **`min_confidence`**, default **60%**), **predicted_class**, **confidence** (percent), and **top_k** ranked alternatives. Paths and **`top_k`** are validated; the model head size must match **`len(class_names)`**. Invalid inputs raise clear **`FileNotFoundError`** / **`ValueError`**; no files are deleted.


In [ ]:
def predict_image(
    image_path,
    model,
    class_names,
    top_k=3,
    min_confidence=60.0,
    show_plot=True,
):
    """
    Returns dict with:
      - status: 'Healthy', 'Diseased', or 'Uncertain' (if top-1 confidence < min_confidence)
      - predicted_class: top-1 class name
      - confidence: top-1 softmax probability in percent [0, 100]
      - top_k: list of {rank, predicted_class, confidence, status} using get_health_status per class
    """
    p = Path(image_path)
    if not p.exists():
        raise FileNotFoundError(f"Image path does not exist: {p.resolve()}")
    if not p.is_file():
        raise ValueError(
            f"Path is not a regular file (e.g. directory or device): {p.resolve()}"
        )

    if top_k < 1:
        raise ValueError(f"top_k must be >= 1, got {top_k!r}.")

    _out = model.output_shape
    if _out is None:
        raise ValueError(
            "model.output_shape is None — build or load the model before predict_image."
        )
    _n_out = _out[-1]
    if _n_out is None:
        raise ValueError(
            f"Cannot validate class count: last dim of output_shape is None (full shape: {_out})."
        )
    if int(_n_out) != len(class_names):
        raise ValueError(
            f"Model output size ({int(_n_out)}) != len(class_names) ({len(class_names)}). "
            "Use the class list from training or class_names.json for this checkpoint."
        )

    try:
        img = load_img(p, target_size=IMG_SIZE)
    except Exception as e:
        raise ValueError(
            f"Could not load image (unsupported format or corrupt file): {p.resolve()}"
        ) from e

    img_array = np.expand_dims(img_to_array(img), axis=0)

    probs = model.predict(img_array, verbose=0)[0]
    if len(probs) != len(class_names):
        raise ValueError(
            f"Prediction vector length ({len(probs)}) != len(class_names) ({len(class_names)})."
        )

    k = min(top_k, len(class_names))
    order = np.argsort(probs)[-k:][::-1]

    predicted_index = int(order[0])
    predicted_class = class_names[predicted_index]
    confidence = float(probs[predicted_index] * 100)

    if confidence < float(min_confidence):
        status = "Uncertain"
    else:
        status = get_health_status(predicted_class)

    top_k_list = []
    for rank, idx in enumerate(order, start=1):
        cname = class_names[int(idx)]
        top_k_list.append(
            {
                "rank": rank,
                "predicted_class": cname,
                "confidence": float(probs[idx] * 100),
                "status": get_health_status(cname),
            }
        )

    if show_plot:
        plt.imshow(img)
        plot_title = (
            f"Status: {status}\nClass: {predicted_class}\n"
            f"Confidence: {confidence:.2f}% (min for label: {float(min_confidence):.1f}%)"
        )
        plt.title(plot_title)
        plt.axis("off")
        plt.show()

    return {
        "status": status,
        "predicted_class": predicted_class,
        "confidence": confidence,
        "top_k": top_k_list,
    }


# Example (uncomment): pass best_model after training
# result = predict_image("/content/sample_leaf.jpg", best_model, class_names)
# print(result)


## 17. Save the trained model, class names, and evaluation artifacts

This section writes the **best validation** `.keras` model, **`class_names.json`**, **`model_metadata.json`**, plus **test metrics**, **training history** (head + fine-tune combined), **classification report**, and **raw / row-normalized confusion matrices** as JSON/CSV for reproducibility and reporting. Re-run the **evaluation** cells first so all variables exist.


In [ ]:
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

keras_model_path = MODEL_SAVE_DIR / KERAS_MODEL_FILENAME
class_names_path = MODEL_SAVE_DIR / CLASS_NAMES_FILENAME
metadata_path = MODEL_SAVE_DIR / METADATA_FILENAME
metrics_json_path = MODEL_SAVE_DIR / METRICS_FILENAME
training_history_json_path = MODEL_SAVE_DIR / TRAINING_HISTORY_FILENAME
classification_report_csv_path = MODEL_SAVE_DIR / CLASSIFICATION_REPORT_CSV
confusion_matrix_csv_path = MODEL_SAVE_DIR / CONFUSION_MATRIX_CSV
normalized_confusion_matrix_csv_path = MODEL_SAVE_DIR / CONFUSION_MATRIX_NORM_CSV


def _to_json_scalar(x):
    """Convert NumPy / TF scalars to plain Python for json.dump."""
    if isinstance(x, np.generic):
        return x.item()
    if isinstance(x, (float, int, str, bool)) or x is None:
        return x
    return x


# Final deliverable: best validation model (same as checkpoint on disk).
best_model.save(keras_model_path)

with open(class_names_path, "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2)

metadata = {
    "backbone": "EfficientNetB0",
    "num_classes": int(num_classes),
    "image_size": [int(IMG_SIZE[0]), int(IMG_SIZE[1])],
    "class_names_file": CLASS_NAMES_FILENAME,
    "tensorflow_version": str(tf.__version__),
    "loss": "sparse_categorical_crossentropy",
    "healthy_rule": "After first '___', condition == 'healthy' => Healthy; else Diseased (see section 1 / predict_image)",
    "checkpoint_source": str(CHECKPOINT_PATH),
    "metrics_file": METRICS_FILENAME,
    "training_history_file": TRAINING_HISTORY_FILENAME,
    "classification_report_csv": CLASSIFICATION_REPORT_CSV,
    "confusion_matrix_csv": CONFUSION_MATRIX_CSV,
    "normalized_confusion_matrix_csv": CONFUSION_MATRIX_NORM_CSV,
}
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

def _finite_json_float(x, default_nan=None):
    v = float(_to_json_scalar(x))
    if np.isnan(v) or np.isinf(v):
        return default_nan
    return v


metrics_payload = {
    "test_loss": _finite_json_float(test_metrics.get("loss", float("nan")), None),
    "test_accuracy": _finite_json_float(
        test_metrics.get("accuracy", float("nan")), None
    ),
    "macro_f1": _to_json_scalar(f1_macro),
    "weighted_f1": _to_json_scalar(f1_weighted),
    "balanced_accuracy": _to_json_scalar(bal_acc),
    "number_of_classes": int(num_classes),
    "image_size": [int(IMG_SIZE[0]), int(IMG_SIZE[1])],
    "model_name": MODEL_DISPLAY_NAME,
}
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

history_for_json = {
    key: [_to_json_scalar(v) for v in values]
    for key, values in combined_history.items()
}
with open(training_history_json_path, "w", encoding="utf-8") as f:
    json.dump(history_for_json, f, indent=2)

df_classification.to_csv(classification_report_csv_path, encoding="utf-8")

pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(
    confusion_matrix_csv_path, encoding="utf-8"
)
pd.DataFrame(cm_norm, index=class_names, columns=class_names).to_csv(
    normalized_confusion_matrix_csv_path, encoding="utf-8"
)

print("Saved files:")
for _p in (
    keras_model_path,
    class_names_path,
    metadata_path,
    metrics_json_path,
    training_history_json_path,
    classification_report_csv_path,
    confusion_matrix_csv_path,
    normalized_confusion_matrix_csv_path,
):
    print(" ", _p.resolve())


## 18. Reload verification (saved artifacts)

Sanity-check that the **`.keras` file** and **`class_names.json`** just written can be **reloaded** and that the **output width** matches the **number of labels**. One **batch** from **`test_data`** is run through the reloaded model; failures raise a **clear error** before deployment.


In [ ]:
try:
    loaded_model = tf.keras.models.load_model(str(keras_model_path))
except Exception as e:
    raise RuntimeError(
        f"Could not load saved Keras model from {keras_model_path.resolve()}: {e}"
    ) from e

try:
    with open(class_names_path, "r", encoding="utf-8") as f:
        loaded_class_names = json.load(f)
except Exception as e:
    raise RuntimeError(
        f"Could not read class_names.json from {class_names_path.resolve()}: {e}"
    ) from e

_loaded_out = loaded_model.output_shape
if _loaded_out is None or _loaded_out[-1] is None:
    raise ValueError(
        f"Loaded model has incomplete output_shape ({_loaded_out}); cannot verify class count."
    )

assert int(_loaded_out[-1]) == len(loaded_class_names), (
    f"Class mismatch: model output dim is {int(_loaded_out[-1])} but "
    f"loaded_class_names has length {len(loaded_class_names)}. "
    "Regenerate class_names.json for this checkpoint or use the matching .keras file."
)

_batch = None
for images, _labels in test_data.take(1):
    _batch = images
    break
if _batch is None:
    raise RuntimeError(
        "test_data.take(1) returned no batch — cannot run reload inference check."
    )

try:
    _reload_preds = loaded_model.predict(_batch, verbose=0)
except Exception as e:
    raise RuntimeError(
        f"Forward pass on reloaded model failed for one test batch: {e}"
    ) from e

if int(_reload_preds.shape[-1]) != len(loaded_class_names):
    raise ValueError(
        f"Prediction last dimension ({int(_reload_preds.shape[-1])}) != len(loaded_class_names) "
        f"({len(loaded_class_names)})."
    )

print(
    "Reload verification succeeded: model and class_names.json load correctly, "
    "output_shape matches label count, and one test batch runs through the loaded model."
)


## 19. Real-world performance vs benchmark test accuracy

Strong **benchmark test accuracy** does **not** guarantee good results on **new phone photos**. Dataset images are often clean and similar in style; models can exploit shortcuts that fail outdoors.

### External phone-photo testing

Collect images **outside** the train/test folders. Vary **lighting**, **motion blur**, **shadows**, **background clutter**, and **leaf pose**. Report a few predictions with class, confidence, and Healthy/Diseased.

### Limitations to discuss in your report

- New environments, species, or diseases not seen in training
- Multiple leaves, occlusion, soil, non-leaf objects
- Overconfident wrong predictions on out-of-distribution images

For a **repeatable spot check**, use **section 20** after training: drop images into **`PHONE_TEST_DIR`** (see section 1) and run the batch prediction table.


## 20. Real-world phone-photo spot check

**Why test phone photos?** The **held-out benchmark test set** is often **similar in style** to training (studio-like leaves, controlled backgrounds). **Field and handset images** introduce **blur, glare, shadows, distance, clutter, and pose** that **validation/test accuracy do not measure**. High **benchmark** scores can **overstate** how well the model will work for **real users**; this section is a **qualitative** sanity check on **out-of-folder** images.

Set **`PHONE_TEST_DIR`** in section 1 (default: `phone_test_photos` under **`PROJECT_ROOT`** on Drive). Add **`.jpg` / `.png`** (etc.) images, then run the code cell below. It calls **`predict_image`** with **`show_plot=False`** and summarizes **status**, **class**, **confidence**, and **top-3** candidates.


In [ ]:
_PHONE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if not PHONE_TEST_DIR.exists():
    print(
        "PHONE_TEST_DIR does not exist yet:\n  ",
        PHONE_TEST_DIR.resolve(),
        "\n\nCreate this folder (or change PHONE_TEST_DIR in section 1), then copy real phone photos into it. "
        f"Supported extensions: {sorted(_PHONE_EXTS)}. "
        "Re-run this cell after adding images.",
        sep="",
    )
elif not PHONE_TEST_DIR.is_dir():
    print("PHONE_TEST_DIR is not a directory:", PHONE_TEST_DIR.resolve())
else:
    _phone_paths = sorted(
        {
            p
            for p in PHONE_TEST_DIR.rglob("*")
            if p.is_file() and p.suffix.lower() in _PHONE_EXTS
        },
        key=lambda x: str(x).lower(),
    )
    if not _phone_paths:
        print(
            "No image files found under:\n  ",
            PHONE_TEST_DIR.resolve(),
            "\n\nAdd photos with extensions ",
            sorted(_PHONE_EXTS),
            " (subfolders are scanned).",
            sep="",
        )
    else:
        _rows = []
        for _pth in _phone_paths:
            try:
                _out = predict_image(
                    _pth,
                    best_model,
                    class_names,
                    top_k=3,
                    show_plot=False,
                )
                _tk = _out["top_k"]
                _row = {
                    "image_path": str(_pth.resolve()),
                    "predicted_status": _out["status"],
                    "predicted_class": _out["predicted_class"],
                    "confidence": _out["confidence"],
                    "top_1_class": _tk[0]["predicted_class"] if _tk else None,
                    "top_1_confidence": _tk[0]["confidence"] if _tk else None,
                    "top_2_class": _tk[1]["predicted_class"] if len(_tk) > 1 else None,
                    "top_2_confidence": _tk[1]["confidence"] if len(_tk) > 1 else None,
                    "top_3_class": _tk[2]["predicted_class"] if len(_tk) > 2 else None,
                    "top_3_confidence": _tk[2]["confidence"] if len(_tk) > 2 else None,
                    "error": None,
                }
            except Exception as _e:
                _row = {
                    "image_path": str(_pth.resolve()),
                    "predicted_status": None,
                    "predicted_class": None,
                    "confidence": None,
                    "top_1_class": None,
                    "top_1_confidence": None,
                    "top_2_class": None,
                    "top_2_confidence": None,
                    "top_3_class": None,
                    "top_3_confidence": None,
                    "error": str(_e),
                }
            _rows.append(_row)
        df_phone_results = pd.DataFrame(_rows)
        print(f"Processed {len(_phone_paths)} image(s) from {PHONE_TEST_DIR.resolve()}\n")
        try:
            display(df_phone_results)
        except NameError:
            print(df_phone_results.to_string(index=False))


## 21. Final summary (report checklist)

Run this after **section 17** (artifacts saved) so paths and metric variables exist. Use the printed block as a **checklist** for tables or appendices in a university report; figures and tables appear in earlier sections.


In [ ]:
print("=" * 72)
print("Final summary — plant disease classifier")
print("=" * 72)
_mk = 30
print("Dataset paths:")
print(f"  {'ZIP archive:':<{_mk}} {ZIP_PATH.resolve()}")
print(f"  {'Extract root:':<{_mk}} {EXTRACT_PATH.resolve()}")
print(f"  {'Training directory:':<{_mk}} {TRAIN_DIR.resolve()}")
print(f"  {'Test directory:':<{_mk}} {TEST_DIR.resolve()}")
print(f"{'Number of classes:':<{_mk}} {int(num_classes)}")
print(f"{'Input image size:':<{_mk}} {int(IMG_SIZE[0])} x {int(IMG_SIZE[1])}")
print(f"{'Model name (export):':<{_mk}} {MODEL_DISPLAY_NAME}")
print()
_ta = test_metrics.get("accuracy")
print(
    f"{'Test accuracy (top-1):':<{_mk}} {float(_ta):.4f}"
    if _ta is not None
    else f"{'Test accuracy (top-1):':<{_mk}} n/a"
)
_t3a = test_metrics.get("top3_accuracy")
print(
    f"{'Test top-3 accuracy:':<{_mk}} {float(_t3a):.4f}"
    if _t3a is not None
    else f"{'Test top-3 accuracy:':<{_mk}} n/a"
)
print(f"{'Test macro F1:':<{_mk}} {float(f1_macro):.4f}")
print(f"{'Test weighted F1:':<{_mk}} {float(f1_weighted):.4f}")
print(f"{'Test balanced accuracy:':<{_mk}} {float(bal_acc):.4f}")

print("Saved artifact paths:")
for _lbl, _pth in [
    ("Keras model", keras_model_path),
    ("Class names", class_names_path),
    ("Run metadata", metadata_path),
    ("Test metrics", metrics_json_path),
    ("Training history", training_history_json_path),
    ("Classification report", classification_report_csv_path),
    ("Confusion matrix", confusion_matrix_csv_path),
    ("Normalized confusion matrix", normalized_confusion_matrix_csv_path),
]:
    print(f"  {_lbl}: {_pth.resolve()}")
print("=" * 72)
print("You can copy the lines above into a report; plots and tables appear in sections 12–15.")
